# OmniCare AI: treino dos modelos cirúrgicos (Endoscapes2023)

Este notebook treina e avalia os dois modelos do cenário cirúrgico:

1. **YOLOv8**: detecta 5 estruturas anatômicas e o instrumento cirúrgico.
2. **Classificador de CVS (ResNet18)**: estima os critérios C1, C2 e C3 em cada frame.

No fim, ele mede se cada vídeo do conjunto de teste chegou ao primeiro clipe com a Visão Crítica de Segurança (CVS) atingida.

## Antes de executar

1. Gere o zip no seu computador com `training/prepare_endoscapes.py`.
2. Envie `endoscapes_omnicare.zip` para `Meu Drive/omnicare/`.
3. Em **Ambiente de execução > Alterar o tipo de ambiente de execução**, escolha **GPU T4**.
4. Use **Executar tudo**. O Colab pedirá permissão para acessar o Drive.

**Licença:** esta cópia da base tem a licença restrita da PhysioNet. Mantenha a pasta do Drive privada, sem compartilhar com outras pessoas, e não publique os frames nem os gráficos do YOLO que mostram imagens.

**Se o Colab desconectar:** execute tudo de novo. O YOLOv8 continua do último checkpoint salvo no Drive.

## 1. Ambiente

In [ ]:
import torch

assert torch.cuda.is_available(), 'Sem GPU. Troque o tipo de ambiente de execução para GPU T4.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
!pip install -q ultralytics==8.4.202

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## 2. Configuração

In [ ]:
from pathlib import Path

DRIVE_DIR = Path('/content/drive/MyDrive/omnicare')
ZIP_PATH = DRIVE_DIR / 'endoscapes_omnicare.zip'
OUTPUT_DIR = DRIVE_DIR / 'resultados'
WORK_DIR = Path('/content/endoscapes_work')  # disco local do Colab, bem mais rápido que o Drive

YOLO_BASE = 'yolov8s.pt'
YOLO_EPOCHS = 100
YOLO_PATIENCE = 25
YOLO_IMGSZ = 640
YOLO_BATCH = 16

CVS_EPOCHS = 15
CVS_BATCH = 32
CVS_SIZE = (224, 398)  # altura, largura: mantém a proporção de 854x480
CVS_LR = 3e-4
CVS_RETRAIN = False  # True treina de novo mesmo se o modelo já existir no Drive

SMOOTH_WINDOW = 3  # média móvel de 3 frames rotulados, cerca de 15 s
DEMO_VIDEOS = [162, 166]  # vídeos de teste: 162 nunca atinge o CVS e 166 atinge no fim
NUM_WORKERS = 2
SEED = 42

In [ ]:
import json
import random
import shutil
import zipfile

import numpy as np
import pandas as pd
import torch

SPLITS = ('train', 'val', 'test')
CRITERIA = ['C1', 'C2', 'C3']
DATA_DIR = WORK_DIR / 'endoscapes_omnicare'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
# Murali et al., 2023 (arXiv:2312.12429), tabelas 2 e 7, conjunto de teste.
# São outros modelos: servem de referência, não de meta exata.
PUBLISHED_DETECTION = {  # mAP@[0,5:0,95] de Faster-RCNN, Cascade-RCNN e Deformable-DETR
    'cystic_plate': [8.0, 7.5, 9.3],
    'calot_triangle': [19.4, 18.4, 19.9],
    'cystic_artery': [12.6, 12.1, 14.9],
    'cystic_duct': [15.5, 13.8, 19.0],
    'gallbladder': [62.9, 61.5, 69.6],
    'tool': [61.5, 62.4, 63.7],
    'geral': [30.0, 29.3, 32.7],
}
PUBLISHED_CVS = {
    'ResNet50': {
        'mAP': {'C1': 42.9, 'C2': 46.5, 'C3': 65.1, 'media': 51.5},
        'bAcc': {'C1': 57.6, 'C2': 63.7, 'C3': 70.2, 'media': 63.8},
    },
    'LG-CVS': {'mAP': {'media': 63.3}, 'bAcc': {'media': 74.9}},
    'SV2LSTG': {'mAP': {'media': 65.3}, 'bAcc': {'media': 70.8}},
}

## 3. Dados

In [ ]:
ZIP_PATH = DRIVE_DIR / 'endoscapes_omnicare.zip'
DATA_DIR = WORK_DIR / 'endoscapes_omnicare'
extracted_marker = WORK_DIR / '.extraido'

print('Arquivos em', DRIVE_DIR)
for path in sorted(DRIVE_DIR.iterdir()):
    if path.is_file():
        print(f'  {path.name}: {path.stat().st_size / 1024**3:.2f} GB')

if not ZIP_PATH.exists():
    raise FileNotFoundError(f'Não encontrei endoscapes_omnicare.zip em {DRIVE_DIR}. Envie o arquivo para essa pasta.')

try:
    archive = zipfile.ZipFile(ZIP_PATH)
except zipfile.BadZipFile as error:
    raise RuntimeError('O zip no Drive está incompleto ou corrompido. Apague-o do Drive e envie de novo.') from error

with archive:
    names = archive.namelist()
    print(f'{ZIP_PATH.name}: {len(names)} itens. Primeiros: {names[:3]}')
    if 'endoscapes_omnicare/manifest.json' not in names:
        raise RuntimeError(
            'Este zip tem o nome certo, mas o conteúdo errado. O correto tem 1,11 GB e cerca de 13 mil itens. '
            'Apague-o do Drive e envie de novo o endoscapes_omnicare.zip da pasta Downloads do seu computador.'
        )
    if not extracted_marker.exists() or not (DATA_DIR / 'manifest.json').exists():
        print('Extraindo o zip para o disco local...')
        shutil.rmtree(WORK_DIR, ignore_errors=True)
        WORK_DIR.mkdir(parents=True, exist_ok=True)
        archive.extractall(WORK_DIR)
        extracted_marker.touch()

manifest = json.loads((DATA_DIR / 'manifest.json').read_text(encoding='utf-8'))
frames_df = pd.read_csv(DATA_DIR / 'cvs_labels.csv')
CVS_THRESHOLD = manifest['cvs_threshold']

for split in SPLITS:
    found = len(list((DATA_DIR / 'images' / split).glob('*.jpg')))
    expected = manifest['frames_cvs'][split]
    assert found == expected, f'{split}: {found} frames, esperados {expected}. Apague {WORK_DIR} e execute de novo.'

print('Frames com CVS:', manifest['frames_cvs'])
print('Frames com caixas:', manifest['frames_det'])

In [ ]:
# Regra do relatório técnico da base: um critério é atingido no vídeo se aparecer em
# ao menos um frame rotulado, e o CVS exige os três, não necessariamente juntos.
def official_video_labels(frames):
    reached = (frames[CRITERIA] >= CVS_THRESHOLD).groupby([frames['split'], frames['video_id']]).any()
    reached['cvs_atingido'] = reached.all(axis=1)
    return reached.astype(int).reset_index()


videos_df = official_video_labels(frames_df)
official_counts = videos_df.groupby('split')['cvs_atingido'].sum().to_dict()
print('Vídeos com CVS:', official_counts, 'de', videos_df.groupby('split').size().to_dict())

if manifest['limit_videos'] is None:
    # Números publicados no relatório técnico: 28,3%, 36,6% e 35,0%.
    assert official_counts == {'train': 34, 'val': 15, 'test': 14}, 'Rótulos por vídeo diferentes dos publicados.'

## 4. YOLOv8: detecção de estruturas e instrumento

O espelhamento horizontal fica desligado porque a posição das estruturas na imagem traz informação anatômica.

Detectar as estruturas finas é difícil nesta base: nos modelos publicados, placa cística, triângulo, artéria e ducto ficam entre 7,5 e 19,9 de mAP, contra 61,5 a 69,6 em vesícula e instrumento. A célula de teste mostra essa comparação.

Os checkpoints ficam no Drive, em `resultados/yolo/train/`.

In [ ]:
from ultralytics import YOLO

YOLO_PROJECT = OUTPUT_DIR / 'yolo'
YOLO_RUN = YOLO_PROJECT / 'train'
YOLO_YAML = str(DATA_DIR / 'dataset.yaml')
YOLO_FINAL = OUTPUT_DIR / 'endoscapes_yolov8s.pt'

last_checkpoint = YOLO_RUN / 'weights' / 'last.pt'
if YOLO_FINAL.exists():
    # O modelo final só é copiado depois que o treino termina.
    print('YOLOv8 já treinado; treino ignorado:', YOLO_FINAL)
else:
    if last_checkpoint.exists():
        print('Retomando do checkpoint:', last_checkpoint)
        YOLO(str(last_checkpoint)).train(resume=True, data=YOLO_YAML)
    else:
        YOLO(YOLO_BASE).train(
            data=YOLO_YAML,
            epochs=YOLO_EPOCHS,
            patience=YOLO_PATIENCE,
            imgsz=YOLO_IMGSZ,
            batch=YOLO_BATCH,
            workers=NUM_WORKERS,
            device=0 if DEVICE == 'cuda' else 'cpu',
            seed=SEED,
            fliplr=0.0,
            project=str(YOLO_PROJECT),
            name='train',
            exist_ok=True,
        )
    shutil.copy2(YOLO_RUN / 'weights' / 'best.pt', YOLO_FINAL)
    print('Modelo salvo em', YOLO_FINAL)

In [ ]:
yolo_model = YOLO(str(YOLO_FINAL))
yolo_test = yolo_model.val(
    data=YOLO_YAML,
    split='test',
    imgsz=YOLO_IMGSZ,
    batch=YOLO_BATCH,
    workers=NUM_WORKERS,
    device=0 if DEVICE == 'cuda' else 'cpu',
    project=str(YOLO_PROJECT),
    name='test',
    exist_ok=True,
)

box = yolo_test.box
yolo_results = {
    'precision': float(box.mp),
    'recall': float(box.mr),
    'mAP50': float(box.map50),
    'mAP50_95': float(box.map),
    'per_class': {
        yolo_test.names[int(class_id)]: {'AP50': float(box.ap50[i]), 'AP50_95': float(box.ap[i])}
        for i, class_id in enumerate(box.ap_class_index)
    },
}
print(json.dumps(yolo_results, indent=2))

detection_comparison = pd.DataFrame({
    'este_modelo': {
        **{name: values['AP50_95'] * 100 for name, values in yolo_results['per_class'].items()},
        'geral': yolo_results['mAP50_95'] * 100,
    },
    'publicado_min': {name: min(values) for name, values in PUBLISHED_DETECTION.items()},
    'publicado_max': {name: max(values) for name, values in PUBLISHED_DETECTION.items()},
}).loc[list(PUBLISHED_DETECTION)]
print(detection_comparison.round(1).to_string())

## 5. Classificador de CVS

Mesmo com as caixas anotadas na base, como se a detecção fosse perfeita, a regra "estrutura presente indica o critério" chega a F1 de só 0,46 para o CVS completo. Por isso os critérios têm um classificador próprio.

- **Entrada:** frame inteiro, 11.090 frames rotulados na base completa.
- **Alvo:** média dos 3 especialistas em cada critério, de 0 a 1.
- **Desbalanceamento:** tratado com `pos_weight`. O modelo é escolhido pela precisão média (AP) na validação.

In [ ]:
from PIL import Image
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    precision_recall_curve,
    precision_recall_fscore_support,
    roc_auc_score,
    roc_curve,
)
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
CVS_FINAL = OUTPUT_DIR / 'cvs_resnet18.pt'

train_transform = transforms.Compose([
    transforms.Resize(CVS_SIZE),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.02),
    transforms.RandomAffine(degrees=5, translate=(0.03, 0.03), scale=(0.95, 1.05)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_transform = transforms.Compose([
    transforms.Resize(CVS_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class CvsFrames(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        with Image.open(DATA_DIR / 'images' / row['split'] / row['file_name']) as image:
            tensor = self.transform(image.convert('RGB'))
        return tensor, torch.tensor(row[CRITERIA].to_numpy(dtype=np.float32))


def split_frames(split):
    return frames_df[frames_df['split'] == split].sort_values(['video_id', 'frame']).reset_index(drop=True)


def binary_labels(df):
    return (df[CRITERIA].to_numpy() >= CVS_THRESHOLD).astype(int)


def build_resnet(pretrained):
    weights = models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
    model = models.resnet18(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, len(CRITERIA))
    return model.to(DEVICE)


@torch.no_grad()
def predict_probs(model, df):
    loader = DataLoader(CvsFrames(df, eval_transform), batch_size=CVS_BATCH, num_workers=NUM_WORKERS)
    model.eval()
    return np.concatenate([torch.sigmoid(model(x.to(DEVICE))).cpu().numpy() for x, _ in loader])


def safe_ap(y_true, score):
    return float(average_precision_score(y_true, score)) if y_true.sum() > 0 else float('nan')

In [ ]:
if CVS_FINAL.exists() and not CVS_RETRAIN:
    print('Modelo de CVS já existe no Drive; treino ignorado:', CVS_FINAL)
else:
    train_df, val_df = split_frames('train'), split_frames('val')
    train_loader = DataLoader(
        CvsFrames(train_df, train_transform), batch_size=CVS_BATCH, shuffle=True,
        num_workers=NUM_WORKERS, drop_last=len(train_df) > CVS_BATCH,
    )

    positives = binary_labels(train_df).sum(axis=0)
    pos_weight = np.clip((len(train_df) - positives) / np.maximum(positives, 1), 1, 10)
    print('pos_weight:', dict(zip(CRITERIA, pos_weight.round(2))))

    model = build_resnet(pretrained=True)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight, dtype=torch.float32, device=DEVICE))
    optimizer = torch.optim.AdamW(model.parameters(), lr=CVS_LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CVS_EPOCHS)
    scaler = torch.amp.GradScaler(DEVICE, enabled=DEVICE == 'cuda')

    best_ap, history = -1.0, []
    for epoch in range(1, CVS_EPOCHS + 1):
        model.train()
        running_loss = 0.0
        for images, targets in train_loader:
            images, targets = images.to(DEVICE), targets.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(DEVICE, enabled=DEVICE == 'cuda'):
                loss = criterion(model(images), targets)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item() * len(images)
        scheduler.step()

        val_probs = predict_probs(model, val_df)
        val_true = binary_labels(val_df)
        val_ap = [safe_ap(val_true[:, i], val_probs[:, i]) for i in range(len(CRITERIA))]
        mean_ap = float(np.nanmean(val_ap)) if not np.all(np.isnan(val_ap)) else 0.0
        history.append({'epoch': epoch, 'train_loss': running_loss / len(train_loader.dataset), 'val_mean_ap': mean_ap})
        print(f'Época {epoch}/{CVS_EPOCHS}: loss={history[-1]["train_loss"]:.4f} AP_val={mean_ap:.4f}')

        if mean_ap > best_ap:
            best_ap = mean_ap
            torch.save({
                'state_dict': model.state_dict(),
                'arch': 'resnet18',
                'image_size': list(CVS_SIZE),
                'mean': IMAGENET_MEAN,
                'std': IMAGENET_STD,
                'criteria': CRITERIA,
                'epoch': epoch,
                'val_mean_ap': mean_ap,
            }, CVS_FINAL)

    pd.DataFrame(history).to_csv(OUTPUT_DIR / 'cvs_historico.csv', index=False)
    print('Melhor AP na validação:', round(best_ap, 4))

## 6. Avaliação do CVS por frame

Os limiares são escolhidos na validação e aplicados uma única vez no teste. Como o CVS completo é raro, a acurácia engana: compare com a linha de base "sempre incompleto".

Os próprios cirurgiões discordam bastante (kappa de 0,38 no CVS e de 0,33, 0,53 e 0,44 em C1, C2 e C3), então não espere um modelo quase perfeito. A tabela final compara com a ResNet50 publicada no relatório técnico da base; o modelo daqui é uma ResNet18.

In [ ]:
cvs_checkpoint = torch.load(CVS_FINAL, map_location=DEVICE, weights_only=True)
cvs_model = build_resnet(pretrained=False)
cvs_model.load_state_dict(cvs_checkpoint['state_dict'])


def with_predictions(split):
    df = split_frames(split)
    probs = predict_probs(cvs_model, df)
    for i, criterion_name in enumerate(CRITERIA):
        df[f'p_{criterion_name}'] = probs[:, i]
        # Média só com frames passados: o sistema pode usá-la durante o vídeo.
        df[f's_{criterion_name}'] = df.groupby('video_id')[f'p_{criterion_name}'].transform(
            lambda s: s.rolling(SMOOTH_WINDOW, min_periods=1).mean()
        )
    return df


def best_f1_threshold(y_true, score):
    if y_true.sum() == 0:
        return 0.5
    precision, recall, thresholds = precision_recall_curve(y_true, score)
    f1 = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-9)
    return float(thresholds[np.argmax(f1)])


def binary_metrics(y_true, y_pred, score):
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary', zero_division=0)
    return {
        'prevalencia': float(np.mean(y_true)),
        'AP': safe_ap(y_true, score),
        'precisao': float(precision),
        'recall': float(recall),
        'F1': float(f1),
        'acuracia_balanceada': float(balanced_accuracy_score(y_true, y_pred)) if len(set(y_true)) > 1 else float('nan'),
    }


val_pred, test_pred = with_predictions('val'), with_predictions('test')
val_true = binary_labels(val_pred)
test_true = binary_labels(test_pred)

frame_thresholds = {c: best_f1_threshold(val_true[:, i], val_pred[f'p_{c}'].to_numpy()) for i, c in enumerate(CRITERIA)}

frame_metrics = {}
for i, c in enumerate(CRITERIA):
    score = test_pred[f'p_{c}'].to_numpy()
    frame_metrics[c] = binary_metrics(test_true[:, i], (score >= frame_thresholds[c]).astype(int), score)

complete_true = test_true.all(axis=1).astype(int)
complete_pred = np.all([test_pred[f'p_{c}'].to_numpy() >= frame_thresholds[c] for c in CRITERIA], axis=0).astype(int)
complete_score = test_pred[[f'p_{c}' for c in CRITERIA]].min(axis=1).to_numpy()
frame_metrics['CVS_completo'] = binary_metrics(complete_true, complete_pred, complete_score)
frame_metrics['CVS_completo']['acuracia_sempre_incompleto'] = float(1 - complete_true.mean())

print('Limiares da validação:', {k: round(v, 3) for k, v in frame_thresholds.items()})
print(pd.DataFrame(frame_metrics).T.round(3).to_string())

cvs_comparison = pd.DataFrame({
    'AP_este_modelo': {c: frame_metrics[c]['AP'] * 100 for c in CRITERIA},
    'AP_ResNet50': {c: PUBLISHED_CVS['ResNet50']['mAP'][c] for c in CRITERIA},
    'bAcc_este_modelo': {c: frame_metrics[c]['acuracia_balanceada'] * 100 for c in CRITERIA},
    'bAcc_ResNet50': {c: PUBLISHED_CVS['ResNet50']['bAcc'][c] for c in CRITERIA},
})
cvs_comparison.loc['media'] = cvs_comparison.mean()
print(cvs_comparison.round(1).to_string())

## 7. Desvio por vídeo

Esta é a anomalia do cenário cirúrgico: **o vídeo chegou ao primeiro clipe sem atingir o CVS**.

A regra vem do relatório técnico da base: um critério é atingido no vídeo se aparecer em ao menos um frame rotulado, e o CVS exige os três, não necessariamente ao mesmo tempo. Nos 201 vídeos, 63 atingem o CVS. A página da PhysioNet cita outro número (84 vídeos), da avaliação de um cirurgião, mas o arquivo `vid_cvs.csv` não faz parte do download.

Os limiares por vídeo são escolhidos na validação, e o resultado a relatar é o do teste. Como o teste tem só 40 vídeos, olhe também o intervalo de confiança.

Os vídeos da base já terminam no primeiro clipe. Em um vídeo inteiro, esse momento precisaria ser informado, porque a base não diferencia os tipos de instrumento.

In [ ]:
PEAK_COLUMNS = [f's_{c}' for c in CRITERIA]


# Junta o rótulo oficial de cada vídeo ao maior valor previsto de cada critério.
def video_table(pred_df):
    peaks = pred_df.groupby('video_id')[PEAK_COLUMNS].max()
    return videos_df.merge(peaks, left_on='video_id', right_index=True)


def youden_threshold(y_true, score):
    if len(set(y_true)) < 2:
        return 0.5
    fpr, tpr, thresholds = roc_curve(y_true, score)
    finite = np.isfinite(thresholds)
    return float(thresholds[finite][np.argmax((tpr - fpr)[finite])])


def with_decisions(table, thresholds):
    margins = np.column_stack([table[f's_{c}'] - thresholds[c] for c in CRITERIA])
    table = table.copy()
    table['pred_cvs_atingido'] = (margins >= 0).all(axis=1).astype(int)
    table['desvio_real'] = 1 - table['cvs_atingido']
    table['desvio_previsto'] = 1 - table['pred_cvs_atingido']
    table['escore_desvio'] = -margins.min(axis=1)  # quanto o critério mais distante ficou abaixo do limiar
    return table


def auc_interval(y_true, score, rounds=2000):
    rng = np.random.default_rng(SEED)
    values = []
    for _ in range(rounds):
        chosen = rng.integers(0, len(y_true), len(y_true))
        if len(set(y_true[chosen])) == 2:
            values.append(roc_auc_score(y_true[chosen], score[chosen]))
    return [float(np.percentile(values, 2.5)), float(np.percentile(values, 97.5))] if values else None


def video_metrics(df):
    y_true, y_pred = df['desvio_real'].to_numpy(), df['desvio_previsto'].to_numpy()
    score = df['escore_desvio'].to_numpy()
    both_classes = len(set(y_true)) == 2
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary', zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        'videos': int(len(df)),
        'desvios_reais': int(y_true.sum()),
        'AUROC': float(roc_auc_score(y_true, score)) if both_classes else float('nan'),
        'AUROC_IC95': auc_interval(y_true, score),
        'precisao': float(precision),
        'recall': float(recall),
        'F1': float(f1),
        'acuracia': float((y_true == y_pred).mean()),
        'acuracia_balanceada': float(balanced_accuracy_score(y_true, y_pred)) if both_classes else float('nan'),
        'acuracia_sempre_desvio': float(y_true.mean()),
        'matriz': {'VP': int(tp), 'FP': int(fp), 'FN': int(fn), 'VN': int(tn)},
    }


val_table = video_table(val_pred)
video_thresholds = {c: youden_threshold(val_table[c].to_numpy(), val_table[f's_{c}'].to_numpy()) for c in CRITERIA}
print('Limiares por vídeo (validação):', {c: round(v, 3) for c, v in video_thresholds.items()})

val_videos = with_decisions(val_table, video_thresholds)
test_videos = with_decisions(video_table(test_pred), video_thresholds)
video_results = {'validacao_usada_nos_limiares': video_metrics(val_videos), 'teste': video_metrics(test_videos)}
test_videos.to_csv(OUTPUT_DIR / 'videos_teste_predicoes.csv', index=False)
print(json.dumps(video_results, indent=2, ensure_ascii=False))

In [ ]:
import matplotlib.pyplot as plt

COLORS = {'C1': '#0072B2', 'C2': '#D55E00', 'C3': '#009E73'}

for video_id in [v for v in DEMO_VIDEOS if v in set(test_pred['video_id'])]:
    group = test_pred[test_pred['video_id'] == video_id]
    outcome = test_videos[test_videos['video_id'] == video_id].iloc[0]
    seconds = (group['frame'] - group['frame'].iloc[0]) / 25  # a numeração avança 25 por segundo
    fig, ax = plt.subplots(figsize=(10, 3.5))
    for c in CRITERIA:
        ax.plot(seconds, group[f's_{c}'], color=COLORS[c], label=f'{c} previsto')
        ax.plot(seconds, group[c], color=COLORS[c], linestyle=':', alpha=0.7, label=f'{c} especialistas')
    ax.set(
        xlabel='Tempo na região anotada (s)', ylabel='Valor de 0 a 1', ylim=(-0.05, 1.05),
        title=f'Vídeo {video_id}: real={"CVS atingido" if outcome["cvs_atingido"] else "desvio"} | previsto={"CVS atingido" if outcome["pred_cvs_atingido"] else "desvio"}',
    )
    ax.legend(ncol=3, fontsize=8)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f'linha_do_tempo_video_{video_id}.png', dpi=150)
    plt.show()

## 8. Salvar resultados e conferir os modelos

In [ ]:
import ultralytics

report = {
    'versoes': {'ultralytics': ultralytics.__version__, 'torch': torch.__version__},
    'base': manifest,
    'yolov8_teste': yolo_results,
    'cvs_limiares': {'por_frame': frame_thresholds, 'por_video': video_thresholds},
    'cvs_teste_por_frame': frame_metrics,
    'desvio_por_video': video_results,
    'videos_com_cvs': official_counts,
    'definicao_desvio': (
        'O vídeo chegou ao primeiro clipe sem que C1, C2 e C3 aparecessem, cada um, em ao menos um frame rotulado '
        '(não necessariamente juntos), com cada critério definido pela maioria de 3 especialistas. '
        'Regra do relatório técnico da base.'
    ),
    'referencias_publicadas': {
        'fonte': 'Murali et al., 2023, arXiv:2312.12429, tabelas 2 e 7; outros modelos, apenas como referência.',
        'deteccao_mAP50_95': PUBLISHED_DETECTION,
        'cvs': PUBLISHED_CVS,
    },
    'limitacoes': [
        'Base de um único centro (IHU-Strasbourg), com 201 pacientes e 22 cirurgiões.',
        'Os cirurgiões discordam entre si: kappa de 0,38 no CVS (C1 0,33; C2 0,53; C3 0,44).',
        'Os vídeos da base terminam no primeiro clipe; em um vídeo inteiro, esse momento precisa ser informado.',
        'A classe tool não diferencia instrumentos, então o sistema não detecta a clipagem em si.',
        'O teste tem só 40 vídeos; considere o intervalo de confiança.',
        'Apoio à decisão, não diagnóstico.',
    ],
}
(OUTPUT_DIR / 'metricas.json').write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding='utf-8')

# Confere os arquivos salvos, recarregando-os do Drive.
sample = split_frames('test').iloc[0]
sample_path = DATA_DIR / 'images' / 'test' / sample['file_name']
detections = YOLO(str(YOLO_FINAL)).predict(str(sample_path), imgsz=YOLO_IMGSZ, verbose=False)[0]
print('YOLOv8 recarregado:', len(detections.boxes), 'detecções em', sample['file_name'])

saved = torch.load(CVS_FINAL, map_location=DEVICE, weights_only=True)
reloaded = build_resnet(pretrained=False)
reloaded.load_state_dict(saved['state_dict'])
print('CVS recarregado:', predict_probs(reloaded, split_frames('test').head(1)).round(3).tolist())

print('\nArquivos em', OUTPUT_DIR)
for path in sorted(OUTPUT_DIR.iterdir()):
    if path.is_file():
        print(f'  {path.name} ({path.stat().st_size / 1024**2:.1f} MB)')

## 9. Próximos passos

Baixe de `Meu Drive/omnicare/resultados/`:

- `endoscapes_yolov8s.pt` e `cvs_resnet18.pt`: coloque em `backend/models/`.
- `metricas.json`, `videos_teste_predicoes.csv` e `linha_do_tempo_video_*.png`: use no relatório técnico.

Use no relatório os números reais de `metricas.json`, mesmo que fiquem abaixo do esperado, comparando com as linhas de base e as referências publicadas. As citações das bases estão no README do projeto. A pasta `resultados/yolo/` contém imagens da base, então não a publique.